# Cost saving opportunity for HCP

This jupyter notebook uses the following HCE table to get CPU utilization and AWS cost dataset:
* uip_iceberg.infra_analytics.internal_hyperforce_util_agg_daily
* uip_iceberg.infra_analytics.hyperforce_aws_cost_allocated_view

Please read [this decision record](https://salesforce.quip.com/9u7CATaZNDz8) to understand how to calculate the utilization and cost.


If P95 CPU Utilization < 35%:

$$
  Saving Opportunity = \left(HCE Pod Cost ($) * 0.5\right) \times \left( 1-\frac{P95 CPU Utilization}{35} \right)
$$



## Initialization

In [1]:
%reload_ext autoreload
%autoreload 2

import os
import sys
module_path = os.path.abspath(os.path.join('../'))
sys.path.insert(0, module_path)

from huron.pod import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
from datetime import datetime, timedelta
from collections import Counter

# If you want to print out SQL Query, set log level to logging.INFO.
logging.getLogger().setLevel(logging.INFO)

## Connect to UIP

### Option 1: UIP Notebook (Recommended)

Create Notebook by following [User Guide - UIP Notebook 2.0](https://confluence.internal.salesforce.com/display/UIP/User+Guide+-+UIP+Notebook+2.0#UserGuideUIPNotebook2.0-Prerequisites).

### Option 2: Local environment

1. Download [cacert.pem](https://git.soma.salesforce.com/Infrastructure-Security/puppet_pki_agent/blob/master/files/prod_ca/cacerts.pem) and save it as `.cacert.pem`
2. Copy the trino access code from [Salesforce trino authenticator](https://bdmpresto-access-server.sfproxy.uip.aws-esvc1-useast2.aws.sfdc.cl/)
3. Update the [.secrets.json](./.secrets.json) file

```json
{
    "username": "your_user_name",
    "access_code": "trino_access_code"
}
```

In [2]:
conn = None

try:
    from huron import connect_huron
    conn = connect_huron()
except:
    from uip_client.trino import TrinoConnection

    conn = TrinoConnection(
        catalog='huron-metrics',
        schema='metrics_hourly'
    )

INFO:root:Using Certificate Authentication...


## 2. Configure environment

By default, this script queries the metrics for `stage`, `esvc`, and `prod` environments. If you want to change the environment, please edit `ENV_TYPE` below.

In [3]:
ENV_TYPE = ['dev', 'test', 'perf','prod', 'stage', 'esvc']

TIMERANGE_FROM = '2025-04-01 00:00:00'
TIMERANGE_TO = '2025-04-30 00:00:00'

# Configure maximum rows for displaying dataframes
pd.options.display.max_rows = 100
# Configure maximum cols for displaying dataframes
pd.options.display.max_columns = 50
pd.options.display.float_format = '{:.3f}'.format

NAMESPACES = ["sam-system", "kube-system"]

# Set default custom filter. filter only sam cluster.
CUSTOM_FILTERS = "k8s_cluster like 'sam-%'"
# e.g. filter out coredns pod
#CUSTOM_FILTERS = "k8s_cluster like 'sam-%' AND k8s_pod_name like 'coredns%'"

In [4]:
# The following template gets the maximum request cpu and memory from kube-stat-metric
SQL_RESOURCE_TEMPLATE = """
with dedupe as (
  SELECT
      _time,
      scope,
      k8s_cluster,
      k8s_namespace,
      k8s_pod_name,
      k8s_container_name,
      element_at(tags,'resource') AS metric_type,
      MAX(max_value) AS value
  FROM huron_iceberg.metrics.metrics_hourly
  WHERE
    scope like '%(scope)s'
    AND metric = 'kube_pod_container_resource_requests'
    AND element_at(tags, 'resource') IN ('cpu', 'memory')
    AND element_at(tags, 'environment_type') in (%(envs)s)
    AND _time between TIMESTAMP '%(timerange_from)s' and TIMESTAMP '%(timerange_to)s'
    %(filters)s
  GROUP BY
      1,2,3,4,5,6,7
), sum_agg as (
  SELECT
      _time,
      scope, 
      k8s_cluster,
      k8s_namespace,
      k8s_pod_name,
    SUM(CASE
         WHEN metric_type = 'cpu'
         THEN value
         ELSE NULL
    END) as pod_request_cpu,
    SUM(CASE
         WHEN metric_type = 'memory'
         THEN value
         ELSE NULL
    END) as pod_request_mem
    FROM dedupe
  GROUP BY 1,2,3,4,5
)
SELECT
  format_datetime(_time, 'yyyyMMdd') as date_key,
  scope, 
  k8s_cluster,
  k8s_namespace,
  substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
  arbitrary(k8s_pod_name) as k8s_pod_name,
  MAX(ROUND(pod_request_cpu, 5)) as request_cpu,
  MAX(pod_request_mem) as request_memory
FROM sum_agg
GROUP BY 1,2,3,4,5
ORDER BY date_key, scope, k8s_cluster, k8s_pod_name asc

"""

df_resource_spec = get_pod_with_resource_quota(conn, SQL_RESOURCE_TEMPLATE, TIMERANGE_FROM, TIMERANGE_TO, ENV_TYPE, HOURLY_METRIC_TABLE, NAMESPACES, CUSTOM_FILTERS)
df_resource_spec['resource_type'] = df_resource_spec.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[0], axis=1)
df_resource_spec['resource_name'] = df_resource_spec.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[1], axis=1)
df_resource_spec['deployment_name'] = df_resource_spec.apply(extract_deployment, axis=1)
df_resource_spec

INFO:root:
with dedupe as (
  SELECT
      _time,
      scope,
      k8s_cluster,
      k8s_namespace,
      k8s_pod_name,
      k8s_container_name,
      element_at(tags,'resource') AS metric_type,
      MAX(max_value) AS value
  FROM huron_iceberg.metrics.metrics_hourly
  WHERE
    scope like 'kube-state-metrics.aws.%'
    AND metric = 'kube_pod_container_resource_requests'
    AND element_at(tags, 'resource') IN ('cpu', 'memory')
    AND element_at(tags, 'environment_type') in ('dev', 'test', 'perf', 'prod', 'stage', 'esvc')
    AND _time between TIMESTAMP '2025-04-01 00:00:00' and TIMESTAMP '2025-04-30 00:00:00'
    AND k8s_namespace in ('sam-system', 'kube-system') AND k8s_cluster like 'sam-%'
  GROUP BY
      1,2,3,4,5,6,7
), sum_agg as (
  SELECT
      _time,
      scope, 
      k8s_cluster,
      k8s_namespace,
      k8s_pod_name,
    SUM(CASE
         WHEN metric_type = 'cpu'
         THEN value
         ELSE NULL
    END) as pod_request_cpu,
    SUM(CASE
         WHEN metric_

,date_key,scope,k8s_cluster,k8s_namespace,pod_group,k8s_pod_name,request_cpu,request_memory,resource_type,resource_name,deployment_name
0,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,aws-load-balancer-controller-59654bb7b8,aws-load-balancer-controller-59654bb7b8-9mbsl,0.500,1073741824.000,replicaset,aws-load-balancer-controller-59654bb7b8,aws-load-balancer-controller
1,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,aws-node,aws-node-k28xz,0.025,67108864.000,daemonset,aws-node,aws-node
2,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,sam-system,cloudwatch-logexport-598816239137,cloudwatch-logexport-598816239137-2,0.220,786432000.000,statefulset,cloudwatch-logexport,cloudwatch-logexport
3,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,coredns-8cd545679,coredns-8cd545679-46q7w,0.100,2147483648.000,replicaset,coredns-8cd545679,coredns
4,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,sam-system,dataplane-test-ds,dataplane-test-ds-59vp2,0.080,492830720.000,daemonset,dataplane-test-ds,dataplane-test-ds
...,...,...,...,...,...,...,...,...,...,...,...
772139,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,kube-system,snapshot-controller-545548484d,snapshot-controller-545548484d-dbfc8,0.200,536870912.000,replicaset,snapshot-controller-545548484d,snapshot-controller
772140,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,kube-system,snapshot-validation-6cd44f5dc5,snapshot-validation-6cd44f5dc5-qx7r7,0.200,536870912.000,replicaset,snapshot-validation-6cd44f5dc5,snapshot-validation
772141,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,kube-system,vertical-pod-autoscaler-admission-controller-6...,vertical-pod-autoscaler-admission-controller-6...,0.250,268435456.000,replicaset,vertical-pod-autoscaler-admission-controller-6...,vertical-pod-autoscaler-admission-controller
772142,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,kube-system,vertical-pod-autoscaler-recommender-64db78b945,vertical-pod-autoscaler-recommender-64db78b945...,0.100,1038683533.000,replicaset,vertical-pod-autoscaler-recommender-64db78b945,vertical-pod-autoscaler-recommender


In [5]:
# grouping by cluster
df_resource_quota_per_deploy = df_resource_spec[['date_key','scope','k8s_cluster','k8s_namespace','deployment_name', 'request_cpu', 'request_memory']].groupby(by=['date_key','scope','k8s_cluster','k8s_namespace','deployment_name', 'request_cpu', 'request_memory']).any().reset_index()
df_resource_quota_per_deploy['date_key'] = pd.to_numeric(df_resource_quota_per_deploy['date_key'])
df_resource_quota_per_deploy

,date_key,scope,k8s_cluster,k8s_namespace,deployment_name,request_cpu,request_memory
0,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,aws-load-balancer-controller,0.500,1073741824.000
1,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,aws-node,0.025,67108864.000
2,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,coredns,0.100,2147483648.000
3,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,ebs-csi-controller,0.700,939524096.000
4,20250401,aws.aws-dev2-uswest2.apiq,sam-processing1,kube-system,ebs-csi-node,0.300,805306368.000
...,...,...,...,...,...,...,...
730967,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,sam-system,fkp-watchdog,0.570,3646947328.000
730968,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,sam-system,host-path-permission-setter,0.001,10485760.000
730969,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,sam-system,kube-hpa-generator,0.570,1474297856.000
730970,20250430,aws.test1-uswest2.unified-engagement,sam-processing1,sam-system,orphaned-volumes-pruner,0.250,792723456.000


In [6]:
# Calculate 
sql_cpu_util = """
WITH util_data AS (
  SELECT * FROM uip_iceberg.infra_analytics.internal_hyperforce_util_agg_daily
  WHERE
    date_key between %(timerange_from)s and %(timerange_to)s
    AND level = 'pod'
    AND metric_name = 'cpu_req_util'
    AND aws_service = 'EC2 - Compute'
    AND k8s_pod_name is not NULL
    AND environment_type IN (%(envs)s)
    %(filters)s
)
SELECT
  date_key,
  concat('aws.', falcon_instance, '.', functional_domain) as scope,
  functional_domain,
  falcon_instance,
  environment_type,
  k8s_namespace,
  k8s_cluster,
  substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
  arbitrary(k8s_pod_name) as k8s_pod_name,
  APPROX_PERCENTILE(util, value_count, 0.99) as p99_cpu,
  APPROX_PERCENTILE(util, value_count, 0.95) as p95_cpu
FROM util_data, UNNEST(value_counts)  AS ckv (util, value_count)
GROUP BY 1,2,3,4,5,6,7,8
ORDER BY date_key asc
"""

df_hce_cpu_util = query_aws_pod_unit_cost(conn, sql_cpu_util, TIMERANGE_FROM, TIMERANGE_TO, ENV_TYPE, NAMESPACES, CUSTOM_FILTERS)
df_hce_cpu_util['resource_type'] = df_hce_cpu_util.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[0], axis=1)
df_hce_cpu_util['resource_name'] = df_hce_cpu_util.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[1], axis=1)
df_hce_cpu_util['deployment_name'] = df_hce_cpu_util.apply(extract_deployment, axis=1)
df_hce_cpu_util.drop('k8s_pod_name', axis=1, inplace=True)
df_hce_cpu_util

INFO:root:
WITH util_data AS (
  SELECT * FROM uip_iceberg.infra_analytics.internal_hyperforce_util_agg_daily
  WHERE
    date_key between 20250401 and 20250430
    AND level = 'pod'
    AND metric_name = 'cpu_req_util'
    AND aws_service = 'EC2 - Compute'
    AND k8s_pod_name is not NULL
    AND environment_type IN ('dev', 'test', 'perf', 'prod', 'stage', 'esvc')
    AND k8s_namespace in ('sam-system', 'kube-system') AND k8s_cluster like 'sam-%'
)
SELECT
  date_key,
  concat('aws.', falcon_instance, '.', functional_domain) as scope,
  functional_domain,
  falcon_instance,
  environment_type,
  k8s_namespace,
  k8s_cluster,
  substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
  arbitrary(k8s_pod_name) as k8s_pod_name,
  APPROX_PERCENTILE(util, value_count, 0.99) as p99_cpu,
  APPROX_PERCENTILE(util, value_count, 0.95) as p95_cpu
FROM util_data, UNNEST(value_counts)  AS ckv (util, value_count)
GROUP BY 1,2,3,4,5,6,7,8
ORDER BY date_key asc



,date_key,scope,functional_domain,falcon_instance,environment_type,k8s_namespace,k8s_cluster,pod_group,p99_cpu,p95_cpu,resource_type,resource_name,deployment_name
0,20250401,aws.aws-prod4-apsoutheast2.bigdata,bigdata,aws-prod4-apsoutheast2,prod,sam-system,sam-processing1,cloudwatch-logexport-851725310990,4.387,3.790,statefulset,cloudwatch-logexport,cloudwatch-logexport
1,20250401,aws.perf1-useast2.security,security,perf1-useast2,test,kube-system,sam-mgmt-truth1,aws-node,25.876,15.000,daemonset,aws-node,aws-node
2,20250401,aws.aws-prod22-apnortheast3.foundation,foundation,aws-prod22-apnortheast3,prod,kube-system,sam-log-mon1,vertical-pod-autoscaler-admission-controller-8...,0.944,0.705,replicaset,vertical-pod-autoscaler-admission-controller-8...,vertical-pod-autoscaler-admission-controller
3,20250401,aws.aws-prod11-saeast1.uengage1,uengage1,aws-prod11-saeast1,prod,kube-system,sam-processing1,cluster-autoscaler-6f6879d455,5.813,4.967,replicaset,cluster-autoscaler-6f6879d455,cluster-autoscaler
4,20250401,aws.aws-giaprod1-usgoveast1.einstein2,einstein2,aws-giaprod1-usgoveast1,prod,kube-system,sam-restricted1,vertical-pod-autoscaler-admission-controller-8...,1.000,0.771,replicaset,vertical-pod-autoscaler-admission-controller-8...,vertical-pod-autoscaler-admission-controller
...,...,...,...,...,...,...,...,...,...,...,...,...,...
708586,20250430,aws.aws-prod19-eusouth1.einstein,einstein,aws-prod19-eusouth1,prod,kube-system,sam-processing1,external-dns-fc5f595f6,0.986,0.906,replicaset,external-dns-fc5f595f6,external-dns
708587,20250430,aws.fdev1-uswest2.security,security,fdev1-uswest2,dev,sam-system,sam-mgmt-truth1,kube-hpa-generator-974fc4fff,1.000,1.000,replicaset,kube-hpa-generator-974fc4fff,kube-hpa-generator
708588,20250430,aws.aws-prod19-eusouth1.seas,seas,aws-prod19-eusouth1,prod,sam-system,sam-processing1,dataplane-test-ds,7.000,6.403,daemonset,dataplane-test-ds,dataplane-test-ds
708589,20250430,aws.aws-dev2-uswest2.apiq,apiq,aws-dev2-uswest2,dev,kube-system,sam-restricted1,identity-controller-cred-refresher-5544fcb74d,2.983,2.903,replicaset,identity-controller-cred-refresher-5544fcb74d,identity-controller-cred-refresher


In [7]:
# Calculate 
sql_mem_util = """
WITH util_data AS (
  SELECT * FROM uip_iceberg.infra_analytics.internal_hyperforce_util_agg_daily
  WHERE
    date_key between %(timerange_from)s and %(timerange_to)s
    AND level = 'pod'
    AND metric_name = 'memory_req_util'
    AND aws_service = 'EC2 - Compute'
    AND k8s_pod_name is not NULL
    AND environment_type IN (%(envs)s)
    %(filters)s
)
SELECT
  date_key,
  concat('aws.', falcon_instance, '.', functional_domain) as scope,
  functional_domain,
  falcon_instance,
  environment_type,
  k8s_namespace,
  k8s_cluster,
  substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
  arbitrary(k8s_pod_name) as k8s_pod_name,
  APPROX_PERCENTILE(util, value_count, 0.99) as p99_mem,
  APPROX_PERCENTILE(util, value_count, 0.95) as p95_mem
FROM util_data, UNNEST(value_counts)  AS ckv (util, value_count)
GROUP BY 1,2,3,4,5,6,7,8
ORDER BY date_key asc
"""

df_hce_mem_util = query_aws_pod_unit_cost(conn, sql_mem_util, TIMERANGE_FROM, TIMERANGE_TO, ENV_TYPE, NAMESPACES, CUSTOM_FILTERS)
df_hce_mem_util['resource_type'] = df_hce_mem_util.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[0], axis=1)
df_hce_mem_util['resource_name'] = df_hce_mem_util.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[1], axis=1)
df_hce_mem_util['deployment_name'] = df_hce_mem_util.apply(extract_deployment, axis=1)
df_hce_mem_util.drop('k8s_pod_name', axis=1, inplace=True)
df_hce_mem_util

INFO:root:
WITH util_data AS (
  SELECT * FROM uip_iceberg.infra_analytics.internal_hyperforce_util_agg_daily
  WHERE
    date_key between 20250401 and 20250430
    AND level = 'pod'
    AND metric_name = 'memory_req_util'
    AND aws_service = 'EC2 - Compute'
    AND k8s_pod_name is not NULL
    AND environment_type IN ('dev', 'test', 'perf', 'prod', 'stage', 'esvc')
    AND k8s_namespace in ('sam-system', 'kube-system') AND k8s_cluster like 'sam-%'
)
SELECT
  date_key,
  concat('aws.', falcon_instance, '.', functional_domain) as scope,
  functional_domain,
  falcon_instance,
  environment_type,
  k8s_namespace,
  k8s_cluster,
  substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
  arbitrary(k8s_pod_name) as k8s_pod_name,
  APPROX_PERCENTILE(util, value_count, 0.99) as p99_mem,
  APPROX_PERCENTILE(util, value_count, 0.95) as p95_mem
FROM util_data, UNNEST(value_counts)  AS ckv (util, value_count)
GROUP BY 1,2,3,4,5,6,7,8
ORDER BY date_key asc



,date_key,scope,functional_domain,falcon_instance,environment_type,k8s_namespace,k8s_cluster,pod_group,p99_mem,p95_mem,resource_type,resource_name,deployment_name
0,20250401,aws.aws-prod1-useast1.core1,core1,aws-prod1-useast1,prod,kube-system,sam-restricted6,aws-node,81.511,75.640,daemonset,aws-node,aws-node
1,20250401,aws.dev1-uswest2.sfcdmodel,sfcdmodel,dev1-uswest2,dev,sam-system,sam-processing1,fit-karpenter-smoke-test-29058120,56.000,56.000,unknown,fit-karpenter-smoke-test-29058120-r6kxk,fit-karpenter-smoke-test-29058120-r6kxk
2,20250401,aws.aws-prod22-apnortheast3.core1,core1,aws-prod22-apnortheast3,prod,sam-system,sam-restricted3,fkp-watchdog-7b8cbc897c,3.991,3.911,replicaset,fkp-watchdog-7b8cbc897c,fkp-watchdog
3,20250401,aws.aws-giadev1-usgoveast1.foundation,foundation,aws-giadev1-usgoveast1,dev,sam-system,sam-control1,fkp-watchdog-7587d79885,3.000,3.000,replicaset,fkp-watchdog-7587d79885,fkp-watchdog
4,20250401,aws.aws-dev2-uswest2.foundation,foundation,aws-dev2-uswest2,dev,kube-system,sam-logging-monitoring1,kube-dns-autoscaler-75d4d9d77f,14.000,14.000,replicaset,kube-dns-autoscaler-75d4d9d77f,kube-dns-autoscaler
...,...,...,...,...,...,...,...,...,...,...,...,...,...
715320,20250430,aws.aws-prod5-uswest2.uengage1,uengage1,aws-prod5-uswest2,prod,kube-system,sam-processing1,vertical-pod-autoscaler-admission-controller-7...,53.838,53.082,replicaset,vertical-pod-autoscaler-admission-controller-7...,vertical-pod-autoscaler-admission-controller
715321,20250430,aws.aws-dev5-apsouth1.foundation,foundation,aws-dev5-apsouth1,dev,kube-system,sam-log-mon1,ebs-csi-controller-957fbc6dc,6.000,6.000,replicaset,ebs-csi-controller-957fbc6dc,ebs-csi-controller
715322,20250430,aws.aws-prod13-euwest2.einstein,einstein,aws-prod13-euwest2,prod,kube-system,sam-processing1,external-dns-7fcc8896b6,5.994,5.914,replicaset,external-dns-7fcc8896b6,external-dns
715323,20250430,aws.aws-dev4-uswest2.security,security,aws-dev4-uswest2,dev,sam-system,sam-mgmt-truth1,host-path-permission-setter,8.000,8.000,daemonset,host-path-permission-setter,host-path-permission-setter


In [8]:
sql_cost = """
  SELECT
    date_key,
    concat('aws.', falcon_instance, '.', functional_domain) as scope,
    falcon_instance,
    functional_domain,
    environment_type,
    allocated_service,
    aws_eks_cluster_name as k8s_cluster,
    k8s_namespace,
    substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
    arbitrary(k8s_pod_name) as k8s_pod_name,
    sum(total_cost) as ec2_cost
  FROM uip_iceberg.infra_analytics.hyperforce_aws_cost_allocated_view
  WHERE
    date_key between %(timerange_from)s and %(timerange_to)s
    AND aws_service = 'EC2 - Compute'
    AND k8s_namespace is not NULL AND k8s_container_name is not NULL AND k8s_pod_name is not NULL
    AND environment_type IN (%(envs)s)
    %(filters)s
  GROUP BY
    1,2,3,4,5,6,7,8,9
  ORDER BY date_key
"""

df_cost = query_aws_pod_unit_cost(conn, sql_cost, TIMERANGE_FROM, TIMERANGE_TO, ENV_TYPE, NAMESPACES, CUSTOM_FILTERS)
# The following code parses k8s_pod_name to identify the resource type such as replicaset, statefulset, daemonset, and unknown.
df_cost['resource_type'] = df_cost.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[0], axis=1)
df_cost['resource_name'] = df_cost.apply(lambda row: parse_pod_name(row.get('k8s_pod_name', ''))[1], axis=1)

# This code extracts deployment type name if resource_type is replicaset.
df_cost['deployment_name'] = df_cost.apply(extract_deployment, axis=1)
df_cost.drop('k8s_pod_name', axis=1, inplace=True)
df_cost


INFO:root:
  SELECT
    date_key,
    concat('aws.', falcon_instance, '.', functional_domain) as scope,
    falcon_instance,
    functional_domain,
    environment_type,
    allocated_service,
    aws_eks_cluster_name as k8s_cluster,
    k8s_namespace,
    substr(k8s_pod_name, 1, strpos(k8s_pod_name, '-', -1)-1) as pod_group,
    arbitrary(k8s_pod_name) as k8s_pod_name,
    sum(total_cost) as ec2_cost
  FROM uip_iceberg.infra_analytics.hyperforce_aws_cost_allocated_view
  WHERE
    date_key between 20250401 and 20250430
    AND aws_service = 'EC2 - Compute'
    AND k8s_namespace is not NULL AND k8s_container_name is not NULL AND k8s_pod_name is not NULL
    AND environment_type IN ('dev', 'test', 'perf', 'prod', 'stage', 'esvc')
    AND k8s_namespace in ('sam-system', 'kube-system') AND aws_eks_cluster_name like 'sam-%'
  GROUP BY
    1,2,3,4,5,6,7,8,9
  ORDER BY date_key



,date_key,scope,falcon_instance,functional_domain,environment_type,allocated_service,k8s_cluster,k8s_namespace,pod_group,ec2_cost,resource_type,resource_name,deployment_name
0,20250401,aws.aws-prod11-saeast1.core1,aws-prod11-saeast1,core1,prod,sam,sam-restricted1,kube-system,external-dns-67bdbc4c44,0.434,replicaset,external-dns-67bdbc4c44,external-dns
1,20250401,aws.aws-prod4-apsoutheast2.cdp1,aws-prod4-apsoutheast2,cdp1,prod,sam,sam-processing1,sam-system,dataplane-test-ds,1.261,daemonset,dataplane-test-ds,dataplane-test-ds
2,20250401,aws.aws-prod15-eucentral2.cdp1,aws-prod15-eucentral2,cdp1,prod,sam,sam-processing1,sam-system,host-path-permission-setter,0.028,daemonset,host-path-permission-setter,host-path-permission-setter
3,20250401,aws.aws-prod1-useast1.ecom1,aws-prod1-useast1,ecom1,prod,identity-controller-refresher,sam-processing1,kube-system,identity-controller-cred-refresher-6f8b77c99d,0.011,replicaset,identity-controller-cred-refresher-6f8b77c99d,identity-controller-cred-refresher
4,20250401,aws.aws-esvc1-useast2.buildndeliver,aws-esvc1-useast2,buildndeliver,esvc,sam,sam-processing1,kube-system,cluster-autoscaler-7454df5cd6,0.790,replicaset,cluster-autoscaler-7454df5cd6,cluster-autoscaler
...,...,...,...,...,...,...,...,...,...,...,...,...,...
728955,20250430,aws.aws-prod3-eucentral1.seas,aws-prod3-eucentral1,seas,prod,sam,sam-processing1,kube-system,coredns-7d75f45f59,2.294,replicaset,coredns-7d75f45f59,coredns
728956,20250430,aws.aws-prod4-apsoutheast2.ecom1,aws-prod4-apsoutheast2,ecom1,prod,sam,sam-processing1,kube-system,snapshot-validation-7c9f9668b9,0.176,replicaset,snapshot-validation-7c9f9668b9,snapshot-validation
728957,20250430,aws.aws-prod21-useast2.cdp2,aws-prod21-useast2,cdp2,prod,identitycontrollertest,sam-processing1,kube-system,identity-controller-fit-test-5866ddb4f,0.020,replicaset,identity-controller-fit-test-5866ddb4f,identity-controller-fit-test
728958,20250430,aws.aws-prod24-apsouth2.cdp1,aws-prod24-apsouth2,cdp1,prod,sam,sam-processing1,sam-system,fkp-watchdog-86846ff55d,0.117,replicaset,fkp-watchdog-86846ff55d,fkp-watchdog


In [9]:
df_cost[['date_key', 'ec2_cost']].groupby(by='date_key').sum()
df_cost.to_csv("cpu_cost_aug01_aug31_all_env.csv")

In [10]:
df_cost[df_cost['pod_group'].str.startswith('kube-node-recycler')] 

,date_key,scope,falcon_instance,functional_domain,environment_type,allocated_service,k8s_cluster,k8s_namespace,pod_group,ec2_cost,resource_type,resource_name,deployment_name


In [ ]:
df_util_merged = df_hce_cpu_util.merge(df_hce_mem_util, how='left').merge(df_resource_quota_per_deploy, how='left')
withcost_hce = df_cost.merge(df_util_merged, how='left').sort_values(by=['date_key', 'scope', 'k8s_cluster']).reset_index(drop=True)
withcost_hce
withcost_hce.to_csv("sam_cost_aug01_aug31_all_env.csv")

In [ ]:
withcost_hce[withcost_hce['deployment_name'] == 'coredns']

In [ ]:
# Use the same calculation - https://confluence.internal.salesforce.com/display/DATASCI/Cost+-+Frequently+Asked+Questions#CostFrequentlyAskedQuestions-Campaign:LowP95CPUUtilization

def calculate_saving_cost(row):
    if row['p95_cpu'] >= 35.0:
        return 0.0
    return (row['ec2_cost'] * 0.50) * (1.0-(row['p95_cpu']/35.0))

withcost_hce['saving_opportunity'] = withcost_hce.apply(calculate_saving_cost, axis=1)
withcost_hce
withcost_hce.to_csv("savingsopportunity_aug01_aug31_all_env.csv")

In [ ]:
# filter out unncessary deployments e.g. fit-validator, moncfg
df_result = withcost_hce[(~withcost_hce['deployment_name'].str.startswith('fit-karpenter')) & (~withcost_hce['deployment_name'].str.startswith('fit-validator')) & (~withcost_hce['deployment_name'].str.startswith('moncfg-'))]

In [ ]:
df_result

In [ ]:
withcost_hce[['deployment_name', 'ec2_cost', 'p95_cpu','p95_mem']].groupby('deployment_name').agg({'ec2_cost': np.sum, 'p95_cpu': np.mean, 'p95_mem':np.mean})

In [ ]:
df_result.to_csv('aug01_aug31_all_env_monthly_cost_utilization.csv', index=False)

In [ ]:
only_coredns= withcost_hce[withcost_hce['deployment_name'] == 'coredns'][['date_key','deployment_name', 'ec2_cost']].groupby(by=['date_key', 'deployment_name']).sum()

In [ ]:
only_coredns.to_csv('coredns_daily_aug01_aug31_all_env.csv')

## Generate report

In [ ]:
ax1 = df_result.plot.scatter(x='p95_cpu', y='p95_mem', c='DarkBlue', marker="x", xlim=[0, 100], ylim=[0, 100])

In [ ]:
df_cost_sum = df_result[['date_key', 'ec2_cost', 'p95_cpu', 'p95_mem']]
df_cost_sum = df_cost_sum[~df_cost_sum['p95_cpu'].isna()]
df_cost_sum = df_cost_sum[~df_cost_sum['p95_mem'].isna()]

df_cost_sum.groupby('date_key').apply(lambda x: np.average(x['p95_cpu'], weights=x['ec2_cost'])).mean()

In [ ]:
df_cost_sum.groupby('date_key').apply(lambda x: np.average(x['p95_mem'], weights=x['ec2_cost'])).mean()

In [ ]:
df_utilization_summary = df_result[['resource_type','deployment_name', 'request_cpu', 'request_memory', 'p95_cpu' ,'p95_mem']].groupby(by=['resource_type','deployment_name', 'request_cpu', 'request_memory']).mean().reset_index()
df_utilization_summary.columns = ['resource_type','Deployment Name', 'Request CPU', 'Request Memory', 'Average CPU Utilization (%) at p95 level', 'Average Memory utilization (%) at p95 level']
df_utilization_summary

In [ ]:
# We want to aggregate daily ec2_cost and saving_opportunity by sum. In order to execute this aggregation,
# 1. Select 'date_key', 'k8s_namespace', 'deployment_name', 'ec2_cost', 'saving_opportunity' columns
# 2. Group by 'date_key', 'k8s_namespace', 'deployment_name' with sum aggregation at deployment level.
# 3. Reset the index

df_dailysaving_by_name = df_result[['date_key',  'resource_type', 'deployment_name', 'ec2_cost', 'saving_opportunity']].groupby(by=['date_key',  'resource_type','deployment_name']).sum().sort_values(by=['date_key','resource_type', 'deployment_name']).reset_index()
df_dailysaving_by_name['yearly_saving'] = df_dailysaving_by_name.apply(lambda x: x['saving_opportunity'] * 365.0, axis=1)
df_dailysaving_by_name.columns = ['Date',  'Resource type', 'Name', 'Daily EC2 cost', 'Daily cost saving opportunity (USD)', 'Yearly saving opportunity estimate (USD, daily saving opportunity * 365)']
df_dailysaving_by_name

In [ ]:
df_daily_sum = df_dailysaving_by_name[['Date', 'Daily EC2 cost', 'Daily cost saving opportunity (USD)']].groupby(by='Date').sum().reset_index()
df_daily_sum[['Daily EC2 cost', 'Daily cost saving opportunity (USD)']].mean()

In [ ]:
df_dailysaving_total_by_namespace = df_result[['date_key',  'ec2_cost', 'saving_opportunity']].groupby(by=['date_key', ]).sum().reset_index()
df_dailysaving_total_by_namespace.columns = ['Date',  'Total daily EC2 cost (USD)', 'Total daily cost saving opportunity (USD)']
df_dailysaving_total_by_namespace

## Export data

In [ ]:
# Export dataframe to csv files
df_utilization_summary.to_csv('./hce-costanalysis-utilization-summary-aug01_aug31_all_env.csv', index=False)
df_dailysaving_by_name.to_csv('./hce-costanalysis-cost-saving-aug01_aug31_all_env.csv', index=False)
df_dailysaving_total_by_namespace.to_csv('./hce-costanalysis-total-saving-namespace-aug01_aug31_all_env.csv', index=False)

In [ ]:
withcost_hce[withcost_hce['deployment_name'] == 'ebs-csi-controller']

In [ ]:
only_ebscsicontroller= withcost_hce[withcost_hce['deployment_name'] == 'ebs-csi-controller'][['date_key','deployment_name', 'ec2_cost']].groupby(by=['date_key', 'deployment_name']).sum()

In [ ]:
only_ebscsicontroller.to_csv('ebs-csi-controller_daily_aug01_aug31_all_env.csv')